# Spark SQL, Catalyst Optimizer y Adaptive Query Execution (AQE)

**Asignatura:** Big Data — Ciencia de Datos, Universidad Externado de Colombia  
**Entorno:** Jupyter Notebook sobre cluster Hadoop/YARN (1 nodo maestro + 2 nodos trabajadores)  
**Dataset:** Global Power Plant Database — World Resources Institute (WRI)

---

**Objetivo de la sesión**

Al finalizar esta sesión, el estudiante será capaz de:

1. Crear vistas temporales a partir de DataFrames y ejecutar consultas SQL con `spark.sql()`.
2. Comparar la misma operación expresada con la API de DataFrames y con Spark SQL.
3. Interpretar planes de ejecución (`explain()`) identificando las fases lógica y física.
4. Comprender las 4 fases del Catalyst Optimizer (parsing → analysis → optimization → physical planning).
5. Entender el concepto de Adaptive Query Execution (AQE) y cuándo interviene.

> **Nota:** Este notebook utiliza el mismo dataset Global Power Plant Database que se trabajó en la guía asincrónica de "Primeros pasos con PySpark en Google Colab". La diferencia fundamental es que ahora Spark se ejecuta sobre **YARN** en un cluster real de 3 nodos, no en modo local.

---
## Bloque 1 — Setup y carga del dataset

### 1.1. Configuración de la SparkSession sobre YARN

En la guía asincrónica, la SparkSession se configuró con `.master("local[*]")` para ejecutar Spark dentro de Colab. Ahora, al trabajar sobre el cluster Hadoop, se utiliza `.master("yarn")` para que YARN administre los recursos (memoria, CPUs) entre los nodos.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark SQL") \
    .master("yarn") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# spark.sql.adaptive.enabled se declara explícito aunque
# ya es el default desde Spark 3.2 (ver Bloque 5),
# para que la configuración quede visible en el código

# Verificar la configuracion
print("Version de Spark:", spark.version)
print("AppName:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)
print("AQE habilitado:", spark.conf.get("spark.sql.adaptive.enabled"))

Version de Spark: 3.5.7
AppName: Spark SQL
Master: yarn
AQE habilitado: true


### 1.2. Descarga del CSV y carga a HDFS

A diferencia de Colab, donde el archivo se lee directamente desde el sistema de archivos local, en un cluster Hadoop los datos deben estar en **HDFS** para que todos los nodos puedan acceder a ellos. Se descargará el CSV al nodo maestro y luego se copiará a HDFS.

In [2]:
%%bash
# Descargar el CSV al sistema de archivos local del nodo maestro
wget -q https://raw.githubusercontent.com/wri/global-power-plant-database/master/output_database/global_power_plant_database.csv \
    -O /tmp/global_power_plant_database.csv

# Verificar que se descargo correctamente
echo "Primeras 2 lineas del archivo:"
head -2 /tmp/global_power_plant_database.csv
echo ""
echo "Numero total de lineas (incluye encabezados):"
wc -l /tmp/global_power_plant_database.csv

Primeras 2 lineas del archivo:
country,country_long,name,gppd_idnr,capacity_mw,latitude,longitude,primary_fuel,other_fuel1,other_fuel2,other_fuel3,commissioning_year,owner,source,url,geolocation_source,wepp_id,year_of_capacity_data,generation_gwh_2013,generation_gwh_2014,generation_gwh_2015,generation_gwh_2016,generation_gwh_2017,generation_gwh_2018,generation_gwh_2019,generation_data_source,estimated_generation_gwh_2013,estimated_generation_gwh_2014,estimated_generation_gwh_2015,estimated_generation_gwh_2016,estimated_generation_gwh_2017,estimated_generation_note_2013,estimated_generation_note_2014,estimated_generation_note_2015,estimated_generation_note_2016,estimated_generation_note_2017
AFG,Afghanistan,Kajaki Hydroelectric Power Plant Afghanistan,GEODB0040538,33.0,32.3220,65.1190,Hydro,,,,,,GEODB,http://globalenergyobservatory.org,GEODB,1009793,2017,,,,,,,,,123.77,162.90,97.39,137.76,119.50,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1

Numero total de lineas (incluye encabezados):


In [3]:
%%bash
# Crear el directorio en HDFS si no existe
hdfs dfs -mkdir -p /datasets/

# Copiar el archivo a HDFS (si ya existe, se sobreescribe)
hdfs dfs -put -f /tmp/global_power_plant_database.csv /datasets/

# Verificar que el archivo esta en HDFS
echo "Contenido del directorio /datasets/ en HDFS:"
hdfs dfs -ls /datasets/global_power_plant_database.csv

Contenido del directorio /datasets/ en HDFS:
-rw-r--r--   2 hadoop supergroup   11973385 2026-09-23 16:46 /datasets/global_power_plant_database.csv


### 1.3. Lectura del DataFrame desde HDFS

Se lee el CSV directamente desde HDFS. Nótese que la ruta utiliza el prefijo `hdfs:///` (o simplemente `/datasets/...` si el `fs.defaultFS` del cluster apunta a HDFS). En la configuración del cluster, el sistema de almacenamiento por defecto es el HDFS, por lo que cualquiera de estas opciones es funcional.

In [4]:
# Leer el CSV desde HDFS
df = spark.read.csv(
    "hdfs:///datasets/global_power_plant_database.csv",
    header=True,
    inferSchema=True
)

print(f"El DataFrame tiene {df.count()} filas y {len(df.columns)} columnas.")
print()
df.printSchema()

El DataFrame tiene 34936 filas y 36 columnas.

root
 |-- country: string (nullable = true)
 |-- country_long: string (nullable = true)
 |-- name: string (nullable = true)
 |-- gppd_idnr: string (nullable = true)
 |-- capacity_mw: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- primary_fuel: string (nullable = true)
 |-- other_fuel1: string (nullable = true)
 |-- other_fuel2: string (nullable = true)
 |-- other_fuel3: string (nullable = true)
 |-- commissioning_year: double (nullable = true)
 |-- owner: string (nullable = true)
 |-- source: string (nullable = true)
 |-- url: string (nullable = true)
 |-- geolocation_source: string (nullable = true)
 |-- wepp_id: string (nullable = true)
 |-- year_of_capacity_data: integer (nullable = true)
 |-- generation_gwh_2013: double (nullable = true)
 |-- generation_gwh_2014: double (nullable = true)
 |-- generation_gwh_2015: double (nullable = true)
 |-- generation_gwh_2016: double (n

> **Recordatorio:** `inferSchema=True` hace que Spark recorra los datos una vez adicional para inferir los tipos. En producción es preferible definir el esquema con `StructType`, pero para esta sesión es suficiente.

---
## Bloque 2 — De DataFrames a Spark SQL

### 2.1. ¿Qué es Spark SQL?

Spark SQL permite consultar DataFrames usando **sintaxis SQL estándar**. Así como en Apache Hive se escribían consultas HQL sobre tablas almacenadas en el metastore, en Spark SQL se escriben consultas sobre **vistas temporales** registradas en la SparkSession.

La ventaja principal: el código SQL y el código de la API de DataFrames son **optimizados de la misma manera** por el motor interno de Spark (Catalyst Optimizer). No hay diferencia de rendimiento entre ambos — la elección es de estilo y legibilidad.

### 2.2. Crear una vista temporal

Para poder ejecutar SQL sobre un DataFrame, primero se debe registrar como una **vista temporal**. Esto es similar a crear una tabla temporal en Hive, pero la vista solo existe dentro de la SparkSession actual.

In [5]:
# Registrar el DataFrame como vista temporal
df.createOrReplaceTempView("plantas")

# Verificar que la vista existe
print("Vistas temporales registradas:")
print(spark.catalog.listTables())

Vistas temporales registradas:
[Table(name='plantas', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


### 2.3. Consulta 1 — SELECT con WHERE (comparación SQL vs. DataFrames API)

**Pregunta:** ¿Cuáles son las 10 centrales eléctricas de mayor capacidad en Colombia?

A continuación se muestra la misma consulta expresada de dos formas: primero con Spark SQL, luego con la API de DataFrames. Ambas producen el **mismo resultado** y el **mismo plan de ejecución**.

In [6]:
# --- Version SQL ---
resultado_sql = spark.sql("""
    SELECT name, country_long, primary_fuel, capacity_mw
    FROM plantas
    WHERE country = 'COL'
    ORDER BY capacity_mw DESC
    LIMIT 10
""")

print("=== Resultado con Spark SQL ===")
resultado_sql.show(truncate=False)

=== Resultado con Spark SQL ===
+-----------------+------------+------------+-----------+
|name             |country_long|primary_fuel|capacity_mw|
+-----------------+------------+------------+-----------+
|San Carlos       |Colombia    |Hydro       |1246.0     |
|Gavio            |Colombia    |Hydro       |1200.0     |
|Chivor           |Colombia    |Hydro       |1000.0     |
|Termobarranquilla|Colombia    |Gas         |918.0      |
|Sogamoso         |Colombia    |Hydro       |820.0      |
|Termoflores      |Colombia    |Gas         |670.0      |
|Porce III        |Colombia    |Hydro       |660.0      |
|Central Guatapé  |Colombia    |Hydro       |560.0      |
|La Sierra        |Colombia    |Gas         |460.0      |
|Miel I           |Colombia    |Hydro       |396.0      |
+-----------------+------------+------------+-----------+



In [7]:
from pyspark.sql.functions import col

# --- Version DataFrames API ---
resultado_df = (
    df.filter(col("country") == "COL")
    .select("name", "country_long", "primary_fuel", "capacity_mw")
    .orderBy(col("capacity_mw").desc())
    .limit(10)
)

print("=== Resultado con DataFrames API ===")
resultado_df.show(truncate=False)

=== Resultado con DataFrames API ===
+-----------------+------------+------------+-----------+
|name             |country_long|primary_fuel|capacity_mw|
+-----------------+------------+------------+-----------+
|San Carlos       |Colombia    |Hydro       |1246.0     |
|Gavio            |Colombia    |Hydro       |1200.0     |
|Chivor           |Colombia    |Hydro       |1000.0     |
|Termobarranquilla|Colombia    |Gas         |918.0      |
|Sogamoso         |Colombia    |Hydro       |820.0      |
|Termoflores      |Colombia    |Gas         |670.0      |
|Porce III        |Colombia    |Hydro       |660.0      |
|Central Guatapé  |Colombia    |Hydro       |560.0      |
|La Sierra        |Colombia    |Gas         |460.0      |
|Miel I           |Colombia    |Hydro       |396.0      |
+-----------------+------------+------------+-----------+



**Observación:** Ambos resultados son idénticos. Más adelante (Bloque 4) se demostrará, con una consulta equivalente, que ambas formas comparten el mismo plan de ejecución interno.

### 2.4. Consulta 2 — GROUP BY con agregación

**Pregunta:** ¿Cuál es la capacidad total instalada por tipo de combustible a nivel mundial?

Esta es la misma pregunta analítica que se resolvió en la guía asincrónica con la API de DataFrames. Ahora se resuelve también con SQL.

In [8]:
# --- Version SQL ---
capacidad_sql = spark.sql("""
    SELECT primary_fuel,
           COUNT(*) AS num_plantas,
           ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    GROUP BY primary_fuel
    ORDER BY capacidad_total_mw DESC
""")

print("=== GROUP BY con Spark SQL ===")
capacidad_sql.show(truncate=False)

=== GROUP BY con Spark SQL ===
+--------------+-----------+------------------+
|primary_fuel  |num_plantas|capacidad_total_mw|
+--------------+-----------+------------------+
|Coal          |2330       |1965541.0         |
|Gas           |3998       |1493050.6         |
|Hydro         |7156       |1053159.62        |
|Nuclear       |195        |407911.76         |
|Wind          |5344       |263053.73         |
|Oil           |2320       |261878.71         |
|Solar         |10665      |188312.32         |
|Biomass       |1430       |34281.3           |
|Waste         |1068       |14748.71          |
|Geothermal    |189        |12687.75          |
|Cogeneration  |41         |4048.0            |
|Other         |43         |3612.86           |
|Petcoke       |12         |2424.58           |
|Storage       |135        |1712.3            |
|Wave and Tidal|10         |552.2             |
+--------------+-----------+------------------+



In [9]:
from pyspark.sql.functions import sum as spark_sum, count, round as spark_round

# --- Version DataFrames API ---
capacidad_df = (
    df.groupBy("primary_fuel")
    .agg(
        count("*").alias("num_plantas"),
        spark_round(spark_sum("capacity_mw"), 2).alias("capacidad_total_mw")
    )
    .orderBy(col("capacidad_total_mw").desc())
)

print("=== GROUP BY con DataFrames API ===")
capacidad_df.show(truncate=False)

=== GROUP BY con DataFrames API ===
+--------------+-----------+------------------+
|primary_fuel  |num_plantas|capacidad_total_mw|
+--------------+-----------+------------------+
|Coal          |2330       |1965541.0         |
|Gas           |3998       |1493050.6         |
|Hydro         |7156       |1053159.62        |
|Nuclear       |195        |407911.76         |
|Wind          |5344       |263053.73         |
|Oil           |2320       |261878.71         |
|Solar         |10665      |188312.32         |
|Biomass       |1430       |34281.3           |
|Waste         |1068       |14748.71          |
|Geothermal    |189        |12687.75          |
|Cogeneration  |41         |4048.0            |
|Other         |43         |3612.86           |
|Petcoke       |12         |2424.58           |
|Storage       |135        |1712.3            |
|Wave and Tidal|10         |552.2             |
+--------------+-----------+------------------+



### 2.5. Escritura y lectura en formato Parquet desde HDFS

En la sesión anterior se mencionó que Spark genera directorios con múltiples archivos `part-*.csv` o `part-*.parquet`. Ahora se escribe a HDFS en formato Parquet y se demuestra que Spark SQL puede consultar directamente datos en este formato.

**¿Por qué Parquet?** Es un formato columnar que ofrece compresión eficiente, preserva los tipos de datos y permite optimizaciones como *predicate pushdown* (el filtro se aplica antes de leer todos los datos) y *projection pruning* (solo se leen las columnas necesarias). Estos conceptos se explorarán en detalle en la sesión de optimización.

In [10]:
# Escribir el DataFrame completo a Parquet en HDFS
df.write.parquet("hdfs:///datasets/plantas_parquet", mode="overwrite")

print("Archivo Parquet escrito en HDFS.")

Archivo Parquet escrito en HDFS.


In [11]:
%%bash
# Verificar los archivos Parquet generados en HDFS
echo "Contenido del directorio Parquet en HDFS:"
hdfs dfs -ls /datasets/plantas_parquet/ | head -10

Contenido del directorio Parquet en HDFS:
Found 3 items
-rw-r--r--   2 hadoop supergroup          0 2026-09-23 16:54 /datasets/plantas_parquet/_SUCCESS
-rw-r--r--   2 hadoop supergroup    1674314 2026-09-23 16:54 /datasets/plantas_parquet/part-00000-c02f1e6b-c604-49ab-b6e2-e959b95eca76-c000.snappy.parquet
-rw-r--r--   2 hadoop supergroup     848730 2026-09-23 16:54 /datasets/plantas_parquet/part-00001-c02f1e6b-c604-49ab-b6e2-e959b95eca76-c000.snappy.parquet


In [12]:
# Leer desde Parquet
df_parquet = spark.read.parquet("hdfs:///datasets/plantas_parquet")

# Registrar como vista temporal
df_parquet.createOrReplaceTempView("plantas_parquet")

# Consultar directamente sobre Parquet
resultado_parquet = spark.sql("""
    SELECT country_long, COUNT(*) AS num_plantas
    FROM plantas_parquet
    WHERE primary_fuel = 'Solar'
    GROUP BY country_long
    ORDER BY num_plantas DESC
    LIMIT 5
""")

print("=== Top 5 paises con plantas solares (desde Parquet) ===")
resultado_parquet.show(truncate=False)

=== Top 5 paises con plantas solares (desde Parquet) ===
+------------------------+-----------+
|country_long            |num_plantas|
+------------------------+-----------+
|United States of America|3283       |
|China                   |1318       |
|United Kingdom          |1170       |
|India                   |851        |
|France                  |817        |
+------------------------+-----------+



---
## Bloque 3 — Consultas analíticas con Spark SQL

### 3.1. Segundo DataFrame: tabla de continentes

Para demostrar JOINs en Spark SQL (así como se realizaron en Hive con HQL), se crea un pequeño DataFrame que mapea códigos de país a continentes. Esto permite responder preguntas como "¿cuál es la capacidad instalada por continente?".

In [13]:
# Crear DataFrame de continentes manualmente
datos_continentes = [
    ("CHN", "China", "Asia"),
    ("IND", "India", "Asia"),
    ("JPN", "Japan", "Asia"),
    ("KOR", "South Korea", "Asia"),
    ("IDN", "Indonesia", "Asia"),
    ("THA", "Thailand", "Asia"),
    ("VNM", "Vietnam", "Asia"),
    ("USA", "United States of America", "Norteamerica"),
    ("CAN", "Canada", "Norteamerica"),
    ("MEX", "Mexico", "Norteamerica"),
    ("BRA", "Brazil", "Sudamerica"),
    ("COL", "Colombia", "Sudamerica"),
    ("ARG", "Argentina", "Sudamerica"),
    ("CHL", "Chile", "Sudamerica"),
    ("PER", "Peru", "Sudamerica"),
    ("DEU", "Germany", "Europa"),
    ("GBR", "United Kingdom", "Europa"),
    ("FRA", "France", "Europa"),
    ("ESP", "Spain", "Europa"),
    ("ITA", "Italy", "Europa"),
    ("RUS", "Russia", "Europa"),
    ("NGA", "Nigeria", "Africa"),
    ("ZAF", "South Africa", "Africa"),
    ("EGY", "Egypt", "Africa"),
    ("KEN", "Kenya", "Africa"),
    ("AUS", "Australia", "Oceania"),
]

df_continentes = spark.createDataFrame(
    datos_continentes,
    schema=["country_code", "country_name", "continente"]
)

df_continentes.createOrReplaceTempView("continentes")

print("Tabla de continentes creada y registrada como vista temporal.")
df_continentes.show(truncate=False)

Tabla de continentes creada y registrada como vista temporal.


+------------+------------------------+------------+
|country_code|country_name            |continente  |
+------------+------------------------+------------+
|CHN         |China                   |Asia        |
|IND         |India                   |Asia        |
|JPN         |Japan                   |Asia        |
|KOR         |South Korea             |Asia        |
|IDN         |Indonesia               |Asia        |
|THA         |Thailand                |Asia        |
|VNM         |Vietnam                 |Asia        |
|USA         |United States of America|Norteamerica|
|CAN         |Canada                  |Norteamerica|
|MEX         |Mexico                  |Norteamerica|
|BRA         |Brazil                  |Sudamerica  |
|COL         |Colombia                |Sudamerica  |
|ARG         |Argentina               |Sudamerica  |
|CHL         |Chile                   |Sudamerica  |
|PER         |Peru                    |Sudamerica  |
|DEU         |Germany                 |Europa 

### 3.2. Consulta analítica 1 — CASE WHEN para etiquetas descriptivas

**Pregunta:** ¿Cuál es la capacidad total instalada por tipo de combustible, con nombres legibles en español?

El uso de `CASE WHEN` para traducir códigos internos a etiquetas descriptivas es una **buena práctica de presentación de datos**. Los códigos son útiles para el procesamiento, pero los resultados que se comunican a usuarios o stakeholders deben ser comprensibles sin necesidad de consultar un diccionario de datos.

In [14]:
capacidad_etiquetas = spark.sql("""
    SELECT 
        CASE primary_fuel
            WHEN 'Hydro' THEN 'Hidroelectrica'
            WHEN 'Solar' THEN 'Solar'
            WHEN 'Wind' THEN 'Eolica'
            WHEN 'Gas' THEN 'Gas natural'
            WHEN 'Oil' THEN 'Petroleo'
            WHEN 'Coal' THEN 'Carbon'
            WHEN 'Nuclear' THEN 'Nuclear'
            WHEN 'Biomass' THEN 'Biomasa'
            WHEN 'Geothermal' THEN 'Geotermica'
            WHEN 'Waste' THEN 'Residuos'
            WHEN 'Petcoke' THEN 'Coque de petroleo'
            WHEN 'Cogeneration' THEN 'Cogeneracion'
            WHEN 'Storage' THEN 'Almacenamiento'
            WHEN 'Wave and Tidal' THEN 'Mareomotriz'
            ELSE primary_fuel
        END AS tipo_combustible,
        COUNT(*) AS num_plantas,
        ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    GROUP BY primary_fuel
    ORDER BY capacidad_total_mw DESC
""")

print("=== Capacidad por tipo de combustible (etiquetas en espanol) ===")
capacidad_etiquetas.show(truncate=False)

# Nota: el combustible "Other" no aparece en la lista de traducciones (cae en el ELSE),
# por lo que se muestra tal cual viene del dataset original.

=== Capacidad por tipo de combustible (etiquetas en espanol) ===
+-----------------+-----------+------------------+
|tipo_combustible |num_plantas|capacidad_total_mw|
+-----------------+-----------+------------------+
|Carbon           |2330       |1965541.0         |
|Gas natural      |3998       |1493050.6         |
|Hidroelectrica   |7156       |1053159.62        |
|Nuclear          |195        |407911.76         |
|Eolica           |5344       |263053.73         |
|Petroleo         |2320       |261878.71         |
|Solar            |10665      |188312.32         |
|Biomasa          |1430       |34281.3           |
|Residuos         |1068       |14748.71          |
|Geotermica       |189        |12687.75          |
|Cogeneracion     |41         |4048.0            |
|Other            |43         |3612.86           |
|Coque de petroleo|12         |2424.58           |
|Almacenamiento   |135        |1712.3            |
|Mareomotriz      |10         |552.2             |
+----------------

### 3.3. Consulta analítica 2 — Generación eléctrica por país (2017)

**Pregunta:** ¿Cuáles son los 10 países con mayor generación eléctrica reportada en 2017?

Se filtra por registros que tengan datos de generación no nulos.

In [15]:
generacion_2017 = spark.sql("""
    SELECT country_long,
           COUNT(*) AS plantas_con_datos,
           ROUND(SUM(generation_gwh_2017), 2) AS generacion_total_gwh
    FROM plantas
    WHERE generation_gwh_2017 IS NOT NULL
    GROUP BY country_long
    ORDER BY generacion_total_gwh DESC
    LIMIT 10
""")

print("=== Top 10 paises por generacion electrica reportada (2017) ===")
generacion_2017.show(truncate=False)

=== Top 10 paises por generacion electrica reportada (2017) ===
+------------------------+-----------------+--------------------+
|country_long            |plantas_con_datos|generacion_total_gwh|
+------------------------+-----------------+--------------------+
|United States of America|8370             |3529342.46          |
|India                   |428              |1115205.11          |
|France                  |41               |389963.29           |
|Germany                 |71               |264230.12           |
|Australia               |242              |192580.57           |
|Italy                   |90               |126273.18           |
|Spain                   |52               |120108.36           |
|Poland                  |29               |110651.74           |
|Sweden                  |22               |79804.94            |
|Belgium                 |17               |57030.13            |
+------------------------+-----------------+--------------------+



### 3.4. Consulta analítica 3 — JOIN: capacidad por continente

**Pregunta:** ¿Cuál es la capacidad total instalada por continente?

Esta consulta combina la tabla `plantas` con la tabla `continentes` mediante un JOIN, de forma análoga a lo que se realizó en las sesiones de Hive con HQL.

> **Nota:** Solo se incluirán en el resultado los países que estén en la tabla `continentes`. Los países no mapeados quedarán excluidos por el INNER JOIN.

In [16]:
capacidad_continente = spark.sql("""
    SELECT c.continente,
           COUNT(*) AS num_plantas,
           ROUND(SUM(p.capacity_mw), 2) AS capacidad_total_mw,
           ROUND(AVG(p.capacity_mw), 2) AS capacidad_promedio_mw
    FROM plantas p
    JOIN continentes c ON p.country = c.country_code
    GROUP BY c.continente
    ORDER BY capacidad_total_mw DESC
""")

print("=== Capacidad instalada por continente ===")
capacidad_continente.show(truncate=False)

=== Capacidad instalada por continente ===
+------------+-----------+------------------+---------------------+
|continente  |num_plantas|capacidad_total_mw|capacidad_promedio_mw|
+------------+-----------+------------------+---------------------+
|Asia        |7088       |2175750.66        |306.96               |
|Norteamerica|11269      |1410582.65        |125.17               |
|Europa      |7985       |700161.77         |87.68                |
|Sudamerica  |2968       |221334.1          |74.57                |
|Africa      |202        |93352.1           |462.14               |
|Oceania     |486        |65852.02          |135.5                |
+------------+-----------+------------------+---------------------+



### 3.5. Consulta analítica 4 — Distribución por décadas con CASE WHEN

**Pregunta:** ¿Cómo se distribuyen las centrales eléctricas según la década en que fueron puestas en operación?

Se utiliza `CASE WHEN` para crear rangos de décadas a partir de `commissioning_year`.

In [17]:
distribucion_decadas = spark.sql("""
    SELECT 
        CASE 
            WHEN commissioning_year < 1950 THEN 'Antes de 1950'
            WHEN commissioning_year BETWEEN 1950 AND 1969 THEN '1950-1969'
            WHEN commissioning_year BETWEEN 1970 AND 1989 THEN '1970-1989'
            WHEN commissioning_year BETWEEN 1990 AND 1999 THEN '1990-1999'
            WHEN commissioning_year BETWEEN 2000 AND 2009 THEN '2000-2009'
            WHEN commissioning_year BETWEEN 2010 AND 2019 THEN '2010-2019'
            WHEN commissioning_year >= 2020 THEN '2020 en adelante'
            ELSE 'Sin dato'
        END AS rango_decada,
        COUNT(*) AS num_plantas,
        ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    GROUP BY rango_decada
    ORDER BY rango_decada
""")

print("=== Distribucion de centrales por decada de comisionamiento ===")
distribucion_decadas.show(truncate=False)

# Nota: ORDER BY rango_decada ordena alfabeticamente los strings, no cronologicamente.
# Por eso 'Antes de 1950' y 'Sin dato' aparecen al final del resultado.
# Para un orden cronologico se puede ordenar por el valor numerico inicial de cada rango.

=== Distribucion de centrales por decada de comisionamiento ===
+----------------+-----------+------------------+
|rango_decada    |num_plantas|capacidad_total_mw|
+----------------+-----------+------------------+
|1950-1969       |1293       |264373.71         |
|1970-1989       |2365       |843500.83         |
|1990-1999       |1578       |389249.37         |
|2000-2009       |3801       |1152851.79        |
|2010-2019       |7103       |1054268.77        |
|2020 en adelante|142        |5596.0            |
|Antes de 1950   |919        |51489.17          |
|Sin dato        |17735      |1945645.81        |
+----------------+-----------+------------------+



**Observación sobre el ordenamiento:** El resultado anterior muestra los rangos en orden **alfabético** — por eso `'Antes de 1950'` aparece al final (la `A` viene después de los dígitos en el orden de caracteres). Para obtener un orden **cronológico**, se puede agregar una segunda columna `CASE WHEN` que asigne un número entero a cada rango, usar ese número para el `ORDER BY`, y luego eliminarlo del resultado final con `.drop()`.

In [18]:
distribucion_decadas_ordenadas = spark.sql("""
    SELECT 
        CASE 
            WHEN commissioning_year < 1950 THEN 'Antes de 1950'
            WHEN commissioning_year BETWEEN 1950 AND 1969 THEN '1950-1969'
            WHEN commissioning_year BETWEEN 1970 AND 1989 THEN '1970-1989'
            WHEN commissioning_year BETWEEN 1990 AND 1999 THEN '1990-1999'
            WHEN commissioning_year BETWEEN 2000 AND 2009 THEN '2000-2009'
            WHEN commissioning_year BETWEEN 2010 AND 2019 THEN '2010-2019'
            WHEN commissioning_year >= 2020 THEN '2020 en adelante'
            ELSE 'Sin dato'
        END AS rango_decada,
        CASE 
            WHEN commissioning_year < 1950 THEN 1
            WHEN commissioning_year BETWEEN 1950 AND 1969 THEN 2
            WHEN commissioning_year BETWEEN 1970 AND 1989 THEN 3
            WHEN commissioning_year BETWEEN 1990 AND 1999 THEN 4
            WHEN commissioning_year BETWEEN 2000 AND 2009 THEN 5
            WHEN commissioning_year BETWEEN 2010 AND 2019 THEN 6
            WHEN commissioning_year >= 2020 THEN 7
            ELSE 8
        END AS orden,
        COUNT(*) AS num_plantas,
        ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    GROUP BY rango_decada, orden
    ORDER BY orden
""")

# La columna 'orden' es auxiliar: permite ordenar cronologicamente sin depender
# del orden alfabetico de los strings. Se elimina antes de mostrar el resultado.
distribucion_decadas_ordenadas = distribucion_decadas_ordenadas.drop("orden")

print("=== Distribucion de centrales por decada (orden cronologico) ===")
distribucion_decadas_ordenadas.show(truncate=False)

=== Distribucion de centrales por decada (orden cronologico) ===
+----------------+-----------+------------------+
|rango_decada    |num_plantas|capacidad_total_mw|
+----------------+-----------+------------------+
|Antes de 1950   |919        |51489.17          |
|1950-1969       |1293       |264373.71         |
|1970-1989       |2365       |843500.83         |
|1990-1999       |1578       |389249.37         |
|2000-2009       |3801       |1152851.79        |
|2010-2019       |7103       |1054268.77        |
|2020 en adelante|142        |5596.0            |
|Sin dato        |17735      |1945645.81        |
+----------------+-----------+------------------+



---
## Bloque 4 — Query Plans y Catalyst Optimizer

### 4.1. ¿Qué es el Catalyst Optimizer?

El **Catalyst Optimizer** es el motor de optimización interno de Spark SQL. Toda consulta — ya sea escrita en SQL o con la API de DataFrames — pasa por 4 fases antes de ejecutarse:

1. **Parsing (Análisis sintáctico):** El texto SQL (o la cadena de métodos del DataFrame) se convierte en un **árbol de plan lógico sin resolver** (*Unresolved Logical Plan*). En esta fase, Spark solo verifica la sintaxis, pero no sabe si las tablas o columnas existen.

2. **Analysis (Análisis semántico):** Spark consulta el **catálogo** (donde están registradas las vistas temporales, esquemas, tipos de datos) para resolver nombres de tablas y columnas. El resultado es un **plan lógico resuelto** (*Analyzed Logical Plan*).

3. **Logical Optimization (Optimización lógica):** Spark aplica reglas de optimización al plan lógico, como:
   - **Predicate pushdown:** mover filtros lo más cerca posible de la fuente de datos.
   - **Projection pruning:** leer solo las columnas que se necesitan.
   - **Constant folding:** evaluar expresiones constantes en tiempo de compilación.
   - **Simplificación de predicados:** eliminar condiciones redundantes.

4. **Physical Planning (Planificación física):** Spark genera uno o más **planes físicos** posibles (estrategias concretas de ejecución) y selecciona el de menor costo estimado. El plan físico define cómo se leerán los datos, qué tipo de join se usará, cómo se redistribuirán los datos entre nodos, etc.

> **Conexión con Hive:** En Apache Hive, se podía usar `EXPLAIN` para ver el plan de ejecución de una consulta HQL. En Spark, el método `.explain()` cumple una función similar, pero muestra más detalle al incluir las fases del Catalyst.

> **Nota:** La secuencia Parsing → Analysis → Optimization → Physical Planning coincide con las 4 secciones que imprime `explain(True)` (Parsed/Analyzed/Optimized/Physical), que es la referencia operativa de esta sesión. La documentación de Databricks describe las fases internas de Catalyst como Analysis → Logical Optimization → Physical Planning → Code Generation.

### 4.2. Visualizar el plan completo con `explain(True)`

El método `explain(True)` (o `explain(mode="extended")`) muestra las 4 fases del plan: Parsed, Analyzed, Optimized y Physical.

In [19]:
# Tomar la consulta de capacidad por combustible con CASE WHEN
consulta_catalyst = spark.sql("""
    SELECT 
        CASE primary_fuel
            WHEN 'Hydro' THEN 'Hidroelectrica'
            WHEN 'Solar' THEN 'Solar'
            WHEN 'Wind' THEN 'Eolica'
            WHEN 'Gas' THEN 'Gas natural'
            WHEN 'Coal' THEN 'Carbon'
            ELSE primary_fuel
        END AS tipo_combustible,
        COUNT(*) AS num_plantas,
        ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    GROUP BY primary_fuel
    ORDER BY capacidad_total_mw DESC
""")

# Mostrar el plan extendido (4 fases)
print("=== Plan de ejecucion extendido ===")
consulta_catalyst.explain(True)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
=== Plan de ejecucion extendido ===
== Parsed Logical Plan ==
'Sort ['capacidad_total_mw DESC NULLS LAST], true
+- 'Aggregate ['primary_fuel], [CASE WHEN ('primary_fuel = Hydro) THEN Hidroelectrica WHEN ('primary_fuel = Solar) THEN Solar WHEN ('primary_fuel = Wind) THEN Eolica WHEN ('primary_fuel = Gas) THEN Gas natural WHEN ('primary_fuel = Coal) THEN Carbon ELSE 'primary_fuel END AS tipo_combustible#654, 'COUNT(1) AS num_plantas#655, 'ROUND('SUM('capacity_mw), 2) AS capacidad_total_mw#656]
   +- 'UnresolvedRelation [plantas], [], false

== Analyzed Logical Plan ==
tipo_combustible: string, num_plantas: bigint, capacidad_total_mw: double
Sort [capacidad_total_mw#656 DESC NULLS LAST], true
+- Aggregate [primary_fuel#24], [CASE WHEN (primary_fuel#24 = Hydro) THEN Hidroelectrica WHEN (primary_fuel#24 = Solar) THEN Solar WHEN (primary_fu

### 4.3. Guía para leer un plan de ejecución

Al observar el plan físico (*Physical Plan*), se encontrarán operadores como:

| Operador en el plan | Significado | Equivalente conceptual |
|---|---|---|
| **FileScan csv** / **FileScan parquet** | Lectura del archivo fuente | `TABLE SCAN` en Hive |
| **Filter** | Aplicación de condiciones WHERE | `WHERE` en SQL |
| **Project** | Selección de columnas (projection) | `SELECT col1, col2` |
| **HashAggregate** | Agregación con hash | `GROUP BY` + funciones de agregación |
| **Sort** | Ordenamiento | `ORDER BY` |
| **Exchange** | Redistribución de datos entre nodos (*shuffle*) | En Hive EXPLAIN se refleja como frontera entre etapas Map/Reduce, no con el nombre "Exchange" |

Al leer el plan **de abajo hacia arriba**, se sigue el flujo de los datos: primero se leen (*Scan*), luego se filtran (*Filter*), se proyectan (*Project*), se agregan (*Aggregate*), se ordenan (*Sort*).

### 4.4. Predicate pushdown con Parquet

Cuando los datos están en formato Parquet, Spark puede aplicar **predicate pushdown**: empujar el filtro al nivel de lectura del archivo, evitando leer bloques completos que no cumplen la condición. Se observa en el plan como `PushedFilters` dentro del operador `FileScan parquet`.

In [20]:
# Consulta con filtro sobre datos en Parquet
consulta_parquet = spark.sql("""
    SELECT name, country_long, primary_fuel, capacity_mw
    FROM plantas_parquet
    WHERE country = 'COL' AND capacity_mw > 50
    ORDER BY capacity_mw DESC
""")

# Mostrar el plan - observar PushedFilters en el FileScan
print("=== Plan de ejecucion sobre Parquet (observar PushedFilters) ===")
consulta_parquet.explain(True)

=== Plan de ejecucion sobre Parquet (observar PushedFilters) ===
== Parsed Logical Plan ==
'Sort ['capacity_mw DESC NULLS LAST], true
+- 'Project ['name, 'country_long, 'primary_fuel, 'capacity_mw]
   +- 'Filter (('country = COL) AND ('capacity_mw > 50))
      +- 'UnresolvedRelation [plantas_parquet], [], false

== Analyzed Logical Plan ==
name: string, country_long: string, primary_fuel: string, capacity_mw: double
Sort [capacity_mw#383 DESC NULLS LAST], true
+- Project [name#381, country_long#380, primary_fuel#386, capacity_mw#383]
   +- Filter ((country#379 = COL) AND (capacity_mw#383 > cast(50 as double)))
      +- SubqueryAlias plantas_parquet
         +- View (`plantas_parquet`, [country#379,country_long#380,name#381,gppd_idnr#382,capacity_mw#383,latitude#384,longitude#385,primary_fuel#386,other_fuel1#387,other_fuel2#388,other_fuel3#389,commissioning_year#390,owner#391,source#392,url#393,geolocation_source#394,wepp_id#395,year_of_capacity_data#396,generation_gwh_2013#397,generati

> **Observación importante:** Comparar el plan anterior con una consulta equivalente sobre el CSV. Al observar el `FileScan` de cada plan, se notará que **ambos formatos muestran `PushedFilters` no vacíos**. La diferencia real está en otra propiedad: `Batched`.
>
> - **Parquet → `Batched: true`:** Los filtros se aplican a nivel de *row group* (bloques de datos en disco). Spark puede saltarse bloques enteros que no cumplen la condición **antes de leerlos**, reduciendo el I/O físico. Esto es el predicate pushdown real.
> - **CSV → `Batched: false`:** Spark debe leer el archivo completo fila por fila y aplica los filtros en memoria. Los `PushedFilters` que aparecen en el plan son declarativos (le indican al lector qué filtrar), pero no permiten saltarse bloques de disco porque CSV no tiene estructura interna que lo permita.
>
> En producción, esta diferencia es significativa: con tablas de cientos de GB, Parquet puede leer solo una fracción del archivo gracias al predicate pushdown, mientras que CSV obliga a leer todo.

In [24]:
# Misma consulta pero sobre el CSV (vista "plantas")
consulta_csv = spark.sql("""
    SELECT name, country_long, primary_fuel, capacity_mw
    FROM plantas
    WHERE country = 'COL' AND capacity_mw > 50
    ORDER BY capacity_mw DESC
""")

print("=== Plan de ejecucion sobre CSV (Batched: false, filtros en memoria) ===")
consulta_csv.explain(True)

=== Plan de ejecucion sobre CSV (Batched: false, filtros en memoria) ===
== Parsed Logical Plan ==
'Sort ['capacity_mw DESC NULLS LAST], true
+- 'Project ['name, 'country_long, 'primary_fuel, 'capacity_mw]
   +- 'Filter (('country = COL) AND ('capacity_mw > 50))
      +- 'UnresolvedRelation [plantas], [], false

== Analyzed Logical Plan ==
name: string, country_long: string, primary_fuel: string, capacity_mw: double
Sort [capacity_mw#21 DESC NULLS LAST], true
+- Project [name#19, country_long#18, primary_fuel#24, capacity_mw#21]
   +- Filter ((country#17 = COL) AND (capacity_mw#21 > cast(50 as double)))
      +- SubqueryAlias plantas
         +- View (`plantas`, [country#17,country_long#18,name#19,gppd_idnr#20,capacity_mw#21,latitude#22,longitude#23,primary_fuel#24,other_fuel1#25,other_fuel2#26,other_fuel3#27,commissioning_year#28,owner#29,source#30,url#31,geolocation_source#32,wepp_id#33,year_of_capacity_data#34,generation_gwh_2013#35,generation_gwh_2014#36,generation_gwh_2015#37,gene

### 4.5. SQL vs. DataFrames API: el mismo plan físico

Una de las demostraciones más importantes de esta sesión es que Catalyst optimiza ambas interfaces de la misma manera. A continuación se comparan los planes físicos de la misma consulta expresada con SQL y con la API de DataFrames.

In [22]:
# --- Version SQL ---
plan_sql = spark.sql("""
    SELECT primary_fuel,
           COUNT(*) AS num_plantas,
           ROUND(SUM(capacity_mw), 2) AS capacidad_total_mw
    FROM plantas
    WHERE capacity_mw > 100
    GROUP BY primary_fuel
    ORDER BY capacidad_total_mw DESC
""")

print("=== Plan FISICO - Version SQL ===")
plan_sql.explain()  # Solo plan fisico

=== Plan FISICO - Version SQL ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [capacidad_total_mw#675 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(capacidad_total_mw#675 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=724]
      +- HashAggregate(keys=[primary_fuel#24], functions=[count(1), sum(capacity_mw#21)])
         +- Exchange hashpartitioning(primary_fuel#24, 200), ENSURE_REQUIREMENTS, [plan_id=721]
            +- HashAggregate(keys=[primary_fuel#24], functions=[partial_count(1), partial_sum(capacity_mw#21)])
               +- Filter (isnotnull(capacity_mw#21) AND (capacity_mw#21 > 100.0))
                  +- FileScan csv [capacity_mw#21,primary_fuel#24] Batched: false, DataFilters: [isnotnull(capacity_mw#21), (capacity_mw#21 > 100.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[hdfs://hadoop-virtualbox:9000/datasets/global_power_plant_database.csv], PartitionFilters: [], PushedFilters: [IsNotNull(capacity_mw), GreaterThan(capacity_m

In [23]:
from pyspark.sql.functions import sum as spark_sum, count, round as spark_round, col

# --- Version DataFrames API ---
plan_df = (
    df.filter(col("capacity_mw") > 100)
    .groupBy("primary_fuel")
    .agg(
        count("*").alias("num_plantas"),
        spark_round(spark_sum("capacity_mw"), 2).alias("capacidad_total_mw")
    )
    .orderBy(col("capacidad_total_mw").desc())
)

print("=== Plan FISICO - Version DataFrames API ===")
plan_df.explain()  # Solo plan fisico

=== Plan FISICO - Version DataFrames API ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [capacidad_total_mw#724 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(capacidad_total_mw#724 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=751]
      +- HashAggregate(keys=[primary_fuel#24], functions=[count(1), sum(capacity_mw#21)])
         +- Exchange hashpartitioning(primary_fuel#24, 200), ENSURE_REQUIREMENTS, [plan_id=748]
            +- HashAggregate(keys=[primary_fuel#24], functions=[partial_count(1), partial_sum(capacity_mw#21)])
               +- Filter (isnotnull(capacity_mw#21) AND (capacity_mw#21 > 100.0))
                  +- FileScan csv [capacity_mw#21,primary_fuel#24] Batched: false, DataFilters: [isnotnull(capacity_mw#21), (capacity_mw#21 > 100.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[hdfs://hadoop-virtualbox:9000/datasets/global_power_plant_database.csv], PartitionFilters: [], PushedFilters: [IsNotNull(capacity_mw), GreaterThan

> **Conclusión:** Ambos planes físicos son **idénticos** (o equivalentes). El Catalyst Optimizer produce la misma estrategia de ejecución sin importar si la consulta se escribió en SQL o con la API de DataFrames. La elección entre ambas interfaces es una cuestión de **estilo y legibilidad**, no de rendimiento.

> **Nota sobre `isFinalPlan=false`:** Al observar los planes anteriores, se puede notar la línea `AdaptiveSparkPlan isFinalPlan=false`. Esto indica que el plan físico mostrado por `.explain()` es el plan **inicial**, antes de que AQE intervenga en tiempo de ejecución. Si AQE decide modificar alguna estrategia durante la ejecución (por ejemplo, fusionar particiones pequeñas o cambiar el tipo de join), el plan final ejecutado puede diferir del que muestra `.explain()`. Este comportamiento se explorará en el siguiente bloque.

> **Delimitación:** En esta sesión, `.explain()` se introdujo como herramienta para **entender la estructura de un plan** y las fases de Catalyst. El objetivo es aprender a leer un plan: identificar un Scan, un Filter, un Aggregate, un Project. En la **sesión de optimización** se retomará `.explain()` para **comparar estrategias de ejecución** (shuffle joins vs. broadcast joins, particionamiento) y tomar decisiones de optimización.

---
## Bloque 5 — Adaptive Query Execution (AQE)

### 5.1. ¿Qué es AQE?

El Catalyst Optimizer trabaja **antes** de la ejecución: analiza el plan, aplica reglas de optimización y genera un plan físico. Sin embargo, las decisiones de Catalyst se basan en **estadísticas estáticas** (tamaño estimado de tablas, cardinalidad estimada). Estas estimaciones pueden ser imprecisas.

**Adaptive Query Execution (AQE)** complementa a Catalyst optimizando el plan **durante la ejecución**, utilizando estadísticas reales obtenidas al completar cada etapa (*stage*) del job.

AQE fue introducido como experimental en Spark 3.0 y está **habilitado por defecto desde Spark 3.2**.

### 5.2. Las 3 optimizaciones principales de AQE

1. **Coalescing de particiones post-shuffle:** Después de una operación de shuffle (como un `GROUP BY`), los datos quedan distribuidos en muchas particiones. Si varias particiones son muy pequeñas, AQE las fusiona (*coalesce*) para reducir la sobrecarga de gestión de tareas.

2. **Conversión de sort-merge join a broadcast join:** Si durante la ejecución AQE detecta que una de las tablas del JOIN es lo suficientemente pequeña (por debajo de un umbral), convierte automáticamente un sort-merge join en un broadcast join, que es mucho más eficiente.

3. **Optimización de skew joins:** Si AQE detecta que una partición tiene significativamente más datos que las demás (*data skew*), divide esa partición en fragmentos más pequeños para evitar que un solo executor se sobrecargue.

In [25]:
# Verificar la configuracion de AQE
print("=== Configuracion de AQE ===")
print("AQE habilitado:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Coalesce particiones:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
print("Tamaño sugerido por particion (bytes):", spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes"))
# Spark devuelve el valor con el sufijo 'b' (bytes). 67108864b = 64 MB (valor por defecto en Spark 3.x)

=== Configuracion de AQE ===
AQE habilitado: true
Coalesce particiones: true
Tamaño sugerido por particion (bytes): 67108864b


### 5.3. Verificación del comportamiento de AQE con datasets pequeños

Para explorar cuándo interviene el coalescing de AQE, se puede consultar el número de particiones resultantes después de un shuffle. Con AQE habilitado, Spark **podría** reducir automáticamente el número de particiones si detecta que varias de ellas son muy pequeñas.

Sin embargo, con un dataset de aproximadamente 35.000 filas como este, es probable que el número de particiones permanezca en el valor por defecto (200). La nota al final de esta sección explica por qué.

In [26]:
# Ejecutar una consulta que produce un shuffle (GROUP BY)
resultado_shuffle = spark.sql("""
    SELECT country_long, COUNT(*) AS num_plantas
    FROM plantas
    GROUP BY country_long
""")

# Forzar la ejecucion para que AQE intervenga
resultado_shuffle.cache()
resultado_shuffle.count()

# Verificar el numero de particiones del resultado
print(f"Numero de particiones del resultado: {resultado_shuffle.rdd.getNumPartitions()}")
print()
print("Si AQE esta habilitado y había particiones pequenas post-shuffle,")
print("el numero de particiones puede ser menor que el valor por defecto")
print(f"(spark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}).")

# Liberar cache
resultado_shuffle.unpersist()

[Stage 42:=================================================>    (185 + 2) / 200]

Numero de particiones del resultado: 200

Si AQE esta habilitado y había particiones pequenas post-shuffle,
el numero de particiones puede ser menor que el valor por defecto
(spark.sql.shuffle.partitions = 200).


DataFrame[country_long: string, num_plantas: bigint]

> **Nota:** En este dataset de aproximadamente 35.000 filas, el efecto de AQE es modesto porque los datos son pequeños. AQE es especialmente importante en **producción**, donde los datasets tienen millones o miles de millones de filas y las distribuciones pueden ser impredecibles. Las estimaciones estáticas de Catalyst son insuficientes en esos escenarios, y AQE corrige las decisiones de optimización en tiempo real.

---
## Bloque 6 — Resumen y conexión con las siguientes sesiones

### Resumen de conceptos clave

1. **Spark SQL** permite consultar DataFrames con sintaxis SQL estándar mediante vistas temporales (`createOrReplaceTempView`) y `spark.sql()`.

2. **SQL y DataFrames API son equivalentes** en rendimiento: ambos pasan por el mismo proceso de optimización (Catalyst) y generan el mismo plan físico.

3. El **Catalyst Optimizer** transforma toda consulta en 4 fases: Parsing → Analysis → Logical Optimization → Physical Planning. Esto incluye optimizaciones como *predicate pushdown* y *projection pruning*.

4. El método `.explain(True)` permite inspeccionar las 4 fases del plan. Los operadores clave son: FileScan, Filter, Project, HashAggregate, Sort, Exchange.

5. **AQE** complementa a Catalyst optimizando durante la ejecución con estadísticas reales: coalescing de particiones, conversión de joins y manejo de data skew.

### Conexión con las siguientes sesiones

- **MLlib (en una sesión posterior del curso):** Se utilizarán DataFrames y Spark SQL como base para preparar los datos de entrada de modelos de Machine Learning distribuido. Las transformaciones aprendidas (filtros, agregaciones, joins) son el paso previo al feature engineering con pipelines de MLlib.

- **Optimización (dos sesiones posteriores):** Se retomará `.explain()` como herramienta para **comparar estrategias de ejecución**, distribuido en dos sesiones: primero la ejecución sobre el clúster y el uso de la Spark UI como herramienta de diagnóstico, y luego broadcast joins vs. shuffle joins, particionamiento físico de datos y configuración de memoria. El enfoque pasará de "leer el plan" a "usar el plan para tomar decisiones de optimización".

In [ ]:
# Cerrar la SparkSession
spark.stop()
print("SparkSession cerrada correctamente.")